# Harvest Slot 사과 단일 이미지 FastAPI + ngrok 서빙 예시

이 노트북은 학습을 수행하지 않고, 이미 학습된 모델 파일 3개를 Kaggle Notebook에서 불러와 FastAPI API로 서빙하는 예시입니다.

사용자는 사과 사진 1장을 업로드합니다. 서버는 먼저 사진이 `top`에 가까운지 `side`에 가까운지 판단하고, 해당 방향의 등급 모델을 선택해 A/B/C 등급과 신선도 보조 점수를 반환합니다.

이 방식은 실제 운영 배포가 아니라, Kaggle 세션에서 앱 연동을 빠르게 확인하기 위한 임시 테스트/시연용입니다.


## 1. 실행 흐름

```text
모델 파일 3개 준비
-> 추론 코드 로드
-> FastAPI 앱 생성
-> Kaggle 내부에서 uvicorn 실행
-> ngrok 공개 URL 생성
-> 외부 앱/Postman에서 이미지 업로드 테스트
```

필요한 모델 파일:

```text
apple_view_top_middle_side_balanced_resnet18_best.pt
apple_top_grade_resnet18_best.pt
apple_middle_grade_resnet18_best.pt
apple_side_grade_resnet18_best.pt
```


## 2. 패키지 설치

Kaggle Notebook에서 한 번 실행합니다. 이미 설치된 패키지는 그대로 넘어갑니다.


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'fastapi',
    'uvicorn',
    'pyngrok',
    'python-multipart',
    'nest_asyncio',
])


## 3. 기본 import와 실행 환경 확인


In [ ]:
from __future__ import annotations

import shutil
import threading
import uuid
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import nest_asyncio
import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image, ImageFilter
from pyngrok import ngrok
from torch import nn
from torchvision import models, transforms

IS_KAGGLE = Path('/kaggle').exists()
WORKING_ROOT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
KAGGLE_INPUT_ROOT = Path('/kaggle/input')

print('IS_KAGGLE:', IS_KAGGLE)
print('WORKING_ROOT:', WORKING_ROOT)
print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


## 4. 모델 파일 경로 설정

모델 파일은 `/kaggle/working/models` 또는 `/kaggle/input` 아래에서 자동 탐색합니다.

Kaggle에서 모델 파일을 Dataset으로 올렸다면 `/kaggle/input/...` 아래에 있어도 됩니다.


In [ ]:
TARGET_FRUIT = 'apple'
GRADE_SCORE_MAP = {'A': 90.0, 'B': 70.0, 'C': 45.0}
VIEW_CONFIDENCE_THRESHOLD = 0.60
GRADE_CONFIDENCE_THRESHOLD = 0.55
DEFAULT_IMAGE_SIZE = 224
USE_BBOX_CROP = False
BBOX_PADDING_RATIO = 0.04

CHECKPOINT_NAMES = {
    'angle': 'apple_view_top_middle_side_balanced_resnet18_best.pt',
    'top_grade': 'apple_top_grade_resnet18_best.pt',
    'middle_grade': 'apple_middle_grade_resnet18_best.pt',
    'side_grade': 'apple_side_grade_resnet18_best.pt',
}


def find_checkpoint(name: str) -> Path:
    candidates = [
        WORKING_ROOT / 'models' / name,
        WORKING_ROOT / name,
    ]
    if KAGGLE_INPUT_ROOT.exists():
        candidates.extend(KAGGLE_INPUT_ROOT.rglob(name))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'{name} 파일을 찾지 못했습니다. Kaggle Dataset 또는 /kaggle/working/models에 모델 파일을 추가하세요.')


CHECKPOINT_PATHS = {task: find_checkpoint(name) for task, name in CHECKPOINT_NAMES.items()}
for task, path in CHECKPOINT_PATHS.items():
    print(task, '->', path)


## 5. 이미지 전처리와 보조 점수 함수

학습 노트의 추론 전처리와 동일하게 `Resize -> ToTensor -> Normalize`를 사용합니다.


In [ ]:
def load_rgb(image_path: str | Path, image_size: int | None = None, bbox: dict[str, int] | None = None) -> Image.Image:
    image = Image.open(image_path).convert('RGB')
    if bbox and USE_BBOX_CROP:
        w, h = image.size
        pad_x = int((bbox['xmax'] - bbox['xmin']) * BBOX_PADDING_RATIO)
        pad_y = int((bbox['ymax'] - bbox['ymin']) * BBOX_PADDING_RATIO)
        left, top = max(0, bbox['xmin'] - pad_x), max(0, bbox['ymin'] - pad_y)
        right, bottom = min(w, bbox['xmax'] + pad_x), min(h, bbox['ymax'] + pad_y)
        if right > left and bottom > top:
            image = image.crop((left, top, right, bottom))
    if image_size:
        image = image.resize((image_size, image_size))
    return image


def build_transforms(image_size: int = DEFAULT_IMAGE_SIZE, train: bool = False, task: str = 'top_grade'):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ])


def image_quality_metrics(image_path: str | Path, bbox: dict[str, int] | None = None) -> dict[str, float]:
    image = load_rgb(image_path, image_size=DEFAULT_IMAGE_SIZE, bbox=bbox)
    hsv = np.asarray(image.convert('HSV'), dtype=np.float32)
    value = hsv[..., 2] / 255.0
    saturation = hsv[..., 1] / 255.0
    edges = np.asarray(image.convert('L').filter(ImageFilter.FIND_EDGES), dtype=np.float32) / 255.0
    return {
        'brightness': round(float(value.mean()), 4),
        'underexposed_ratio': round(float((value < 0.18).mean()), 4),
        'overexposed_ratio': round(float((value > 0.95).mean()), 4),
        'saturation_mean': round(float(saturation.mean()), 4),
        'blur_score': round(float(edges.std()), 4),
    }


def calculate_color_score(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE, bbox: dict[str, int] | None = None) -> float:
    hsv = np.asarray(load_rgb(image_path, image_size=image_size, bbox=bbox).convert('HSV'), dtype=np.float32)
    saturation = hsv[..., 1] / 255.0
    value = hsv[..., 2] / 255.0
    score = (saturation.mean() * 0.65 + value.mean() * 0.35) * 100.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)


def calculate_roundness_score(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE, bbox: dict[str, int] | None = None) -> float:
    image = load_rgb(image_path, image_size=image_size, bbox=bbox)
    arr = np.asarray(image, dtype=np.float32)
    border = np.concatenate([arr[:8].reshape(-1, 3), arr[-8:].reshape(-1, 3), arr[:, :8].reshape(-1, 3), arr[:, -8:].reshape(-1, 3)], axis=0)
    bg = np.median(border, axis=0)
    distance = np.linalg.norm(arr - bg, axis=2)
    mask = distance > max(18.0, float(distance.mean()))
    ys, xs = np.where(mask)
    if len(xs) < 50:
        return 50.0
    width, height = max(1, xs.max() - xs.min() + 1), max(1, ys.max() - ys.min() + 1)
    aspect_score = min(width, height) / max(width, height)
    fill_ratio = mask.sum() / float(width * height)
    score = (aspect_score * 0.65 + min(fill_ratio / 0.78, 1.0) * 0.35) * 100.0
    return round(float(np.clip(score, 0.0, 100.0)), 2)


def calculate_bruise_probability(image_path: str | Path, image_size: int = DEFAULT_IMAGE_SIZE, bbox: dict[str, int] | None = None) -> float:
    hsv = np.asarray(load_rgb(image_path, image_size=image_size, bbox=bbox).convert('HSV'), dtype=np.float32)
    value = hsv[..., 2] / 255.0
    saturation = hsv[..., 1] / 255.0
    dark_ratio = np.logical_and(value < 0.33, saturation > 0.18).mean()
    return round(float(np.clip(min(1.0, dark_ratio / 0.18), 0.0, 1.0)), 4)


## 6. 모델 로드와 단일 이미지 추론 함수


In [ ]:
def build_resnet18(num_classes: int, pretrained: bool = False) -> nn.Module:
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def load_checkpoint_model(checkpoint_path: Path, device: torch.device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    labels = checkpoint['labels']
    model = build_resnet18(num_classes=len(labels), pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model, checkpoint


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOADED_MODELS = {}
for task, path in CHECKPOINT_PATHS.items():
    model, checkpoint = load_checkpoint_model(path, DEVICE)
    LOADED_MODELS[task] = {'model': model, 'checkpoint': checkpoint}
    print(task, checkpoint.get('labels'), 'loaded from', path)


@dataclass(frozen=True)
class AppleFreshnessSingleResult:
    fruit_type: str
    angle_label: str
    angle_confidence: float
    model_grade: str
    grade_confidence: float
    color_score: float
    roundness_score: float
    bruise_probability: float
    image_quality: dict[str, float]
    freshness_score: float
    model_decision: str
    model_version: str


def calculate_freshness_score(model_grade: str, color_score: float, roundness_score: float, bruise_probability: float) -> float:
    grade_score = GRADE_SCORE_MAP.get(model_grade, GRADE_SCORE_MAP['C'])
    bruise_free_score = max(0.0, min(100.0, (1.0 - bruise_probability) * 100.0))
    score = grade_score * 0.60 + color_score * 0.20 + roundness_score * 0.10 + bruise_free_score * 0.10
    return round(max(0.0, min(100.0, score)), 2)


def retake_reason(result: AppleFreshnessSingleResult) -> str | None:
    if result.angle_confidence < VIEW_CONFIDENCE_THRESHOLD:
        return 'VIEW_CONFIDENCE_LOW'
    return None


def make_single_decision(result: AppleFreshnessSingleResult) -> str:
    if retake_reason(result):
        return 'RETAKE'
    if result.grade_confidence < GRADE_CONFIDENCE_THRESHOLD:
        return 'REVIEW'
    if result.freshness_score < 60.0 or result.bruise_probability >= 0.5:
        return 'HOLD'
    if result.freshness_score >= 80.0 and result.model_grade == 'A':
        return 'PASS'
    return 'REVIEW'


def predict_loaded_model(image_path: str | Path, task: str):
    entry = LOADED_MODELS[task]
    model = entry['model']
    checkpoint = entry['checkpoint']
    labels = checkpoint['labels']
    image_size = int(checkpoint.get('image_size', DEFAULT_IMAGE_SIZE))
    transform_task = checkpoint.get('task', task)
    tensor = build_transforms(image_size=image_size, train=False, task=transform_task)(load_rgb(image_path)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0]
    confidence, index = torch.max(probs, dim=0)
    return labels[int(index.item())], round(float(confidence.item()), 4)


def grade_task_for_angle(angle_label: str) -> str:
    if angle_label == 'top':
        return 'top_grade'
    if angle_label == 'middle':
        return 'middle_grade'
    return 'side_grade'


def predict_apple_single(image_path: str | Path) -> AppleFreshnessSingleResult:
    image_path = Path(image_path)
    angle_label, angle_confidence = predict_loaded_model(image_path, 'angle')
    grade_task = grade_task_for_angle(angle_label)
    model_grade, grade_confidence = predict_loaded_model(image_path, grade_task)

    color_score = calculate_color_score(image_path)
    roundness_score = calculate_roundness_score(image_path)
    bruise_probability = calculate_bruise_probability(image_path)
    freshness_score = calculate_freshness_score(
        model_grade=model_grade,
        color_score=color_score,
        roundness_score=roundness_score,
        bruise_probability=bruise_probability,
    )

    result = AppleFreshnessSingleResult(
        fruit_type=TARGET_FRUIT,
        angle_label=angle_label,
        angle_confidence=angle_confidence,
        model_grade=model_grade,
        grade_confidence=grade_confidence,
        color_score=color_score,
        roundness_score=roundness_score,
        bruise_probability=bruise_probability,
        image_quality=image_quality_metrics(image_path),
        freshness_score=freshness_score,
        model_decision='REVIEW',
        model_version='apple-single-image-top-middle-side-view-balanced-router-v0',
    )
    return AppleFreshnessSingleResult(**{**asdict(result), 'model_decision': make_single_decision(result)})


def action_required_for_decision(result: AppleFreshnessSingleResult) -> str:
    if result.model_decision == 'RETAKE':
        return 'RETAKE'
    if result.model_decision in {'REVIEW', 'HOLD'}:
        return 'OWNER_REVIEW'
    return 'NONE'


def response_message_for_result(result: AppleFreshnessSingleResult) -> str:
    if result.model_decision == 'RETAKE':
        return '이미지 각도 판단이 불확실합니다. 사과가 화면 중앙에 오도록 다시 촬영해주세요.'
    if result.model_decision == 'HOLD':
        return '품질 확인이 필요합니다. 점주가 최종 판단해주세요.'
    if result.model_decision == 'PASS':
        return '모델 기준 출고 가능으로 판단됩니다. 점주 최종 확인이 필요합니다.'
    return '모델 판단을 점주가 확인해주세요.'


def to_api_response(result: AppleFreshnessSingleResult, quality_inspection_id: int | None = None, image_url: str | None = None) -> dict[str, Any]:
    return {
        'data': {
            'quality_inspection_id': quality_inspection_id,
            'fruit_type': result.fruit_type,
            'image_url': image_url,
            'angle_label': result.angle_label,
            'angle_confidence': result.angle_confidence,
            'view_confidence_threshold': VIEW_CONFIDENCE_THRESHOLD,
            'model_grade': result.model_grade,
            'grade_confidence': result.grade_confidence,
            'grade_confidence_threshold': GRADE_CONFIDENCE_THRESHOLD,
            'freshness_score': result.freshness_score,
            'color_score': result.color_score,
            'roundness_score': result.roundness_score,
            'bruise_probability': result.bruise_probability,
            'model_decision': result.model_decision,
            'action_required': action_required_for_decision(result),
            'retake_reason': retake_reason(result),
            'model_version': result.model_version,
            'image_quality': result.image_quality,
        },
        'message': response_message_for_result(result),
        'error': None,
    }


## 7. FastAPI 앱 정의

`POST /owner/quality-inspections`에 이미지 파일 1장을 업로드하면 예측 결과를 반환합니다.


In [ ]:
UPLOAD_DIR = WORKING_ROOT / 'uploads'
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(title='Harvest Slot Apple Freshness API')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)


@app.get('/')
def health_check():
    return {
        'data': {'status': 'running'},
        'message': 'success',
        'error': None,
    }


@app.post('/owner/quality-inspections')
async def predict_quality(image: UploadFile = File(...)):
    suffix = Path(image.filename or '').suffix.lower()
    if suffix not in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}:
        suffix = '.png'
    image_path = UPLOAD_DIR / f'{uuid.uuid4().hex}{suffix}'
    with image_path.open('wb') as buffer:
        shutil.copyfileobj(image.file, buffer)

    result = predict_apple_single(image_path)
    return to_api_response(result, quality_inspection_id=None, image_url=str(image_path))


## 8. ngrok authtoken 설정

이 노트는 ngrok authtoken을 코드에 직접 적지 않고 **Kaggle Secrets**에서 읽어옵니다.

Kaggle Notebook 오른쪽 패널에서 아래처럼 등록하세요.

```text
Add-ons -> Secrets
Name: NGROK_AUTH_TOKEN
Value: 본인의 실제 ngrok authtoken
```

토큰 확인 위치:

```text
https://dashboard.ngrok.com/get-started/your-authtoken
```

주의: Secret의 값에 `NGROK_AUTH_TOKEN`이라는 글자를 넣는 것이 아니라, ngrok 대시보드에서 복사한 실제 긴 authtoken 값을 넣어야 합니다.


In [ ]:
from kaggle_secrets import UserSecretsClient

SECRET_NAME = 'NGROK_AUTH_TOKEN'

user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret(SECRET_NAME)

if not NGROK_AUTH_TOKEN:
    raise ValueError(f'Kaggle Secrets에서 {SECRET_NAME} 값을 찾지 못했습니다.')
if NGROK_AUTH_TOKEN in {SECRET_NAME, '여기에_ngrok_authtoken을_입력하세요'}:
    raise ValueError('Secret 값에 실제 ngrok authtoken이 아니라 placeholder 문자열이 들어 있습니다.')
if len(NGROK_AUTH_TOKEN.strip()) < 20:
    raise ValueError('ngrok authtoken 길이가 너무 짧습니다. 실제 authtoken 값을 확인하세요.')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print('ngrok authtoken 설정 완료')
print('token preview:', NGROK_AUTH_TOKEN[:6] + '...')


## 9. FastAPI 서버와 ngrok 터널 실행

이 셀을 실행하면 Kaggle Notebook 세션 안에서 FastAPI 서버가 실행되고, 외부에서 접근 가능한 ngrok URL이 출력됩니다.

Kaggle 세션이 종료되면 서버와 ngrok URL도 같이 종료됩니다.


In [ ]:
PORT = 8000

nest_asyncio.apply()

try:
    ngrok.kill()
except Exception:
    pass


def run_api():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')


server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

public_tunnel = ngrok.connect(PORT)
PUBLIC_URL = public_tunnel.public_url

print('FastAPI local URL:', f'http://127.0.0.1:{PORT}')
print('ngrok public URL:', PUBLIC_URL)
print('health check:', f'{PUBLIC_URL}/')
print('predict endpoint:', f'{PUBLIC_URL}/owner/quality-inspections')


## 10. API 호출 테스트

Kaggle 안에서 샘플 이미지 경로를 하나 지정해 테스트할 수 있습니다.

로컬 앱이나 Postman에서는 출력된 ngrok URL의 `/owner/quality-inspections`로 `image` 파일을 multipart 업로드하면 됩니다.


In [ ]:
# 예시:
# import requests
# sample_image_path = '/kaggle/input/your-dataset/sample.png'
# with open(sample_image_path, 'rb') as f:
#     response = requests.post(
#         f'{PUBLIC_URL}/owner/quality-inspections',
#         files={'image': f},
#     )
# print(response.status_code)
# print(response.json())


## 정리

이 노트는 학습용이 아니라 서빙 테스트용입니다.

포함된 것:

```text
모델 파일 자동 탐색
ResNet18 모델 구조
이미지 전처리
단일 이미지 추론
FastAPI API
ngrok 터널링
```

포함하지 않는 것:

```text
데이터셋 탐색
JSON 라벨 파싱
train/valid/test 분리
모델 학습
평가 리포트
```

최종 운영 배포는 Kaggle/ngrok보다 별도 서버에 FastAPI를 올리고 모델 파일을 배치하는 방식이 더 안정적입니다.
